# Approach 1
This approach applies TCDB to acquire transporters and their TC identification (TCID). Subsequently, mechanisms for the relevant families are obtained manually, before they are connected to the reaction of the mechanism. This will in turn be crosschecked with Rhea, which is mapped to through UniProt IDs (UID).

Semantically, it will follow something along these lines: TCID + substrate + mechanism -> chemical reaction. Connect this and compare with reaction on Rhea.

In [1]:
import requests
from Bio import SeqIO
from io import StringIO
import pandas as pd
import numpy as np

The relevant files from the Mapping Files on TCDB, are as following:
1) "Tab-delimited table mapping TC uniprot/refseq accessions to TC systems" -tc_uid_url
2) "Tab-delimited table mapping TC systems to their substrates and ChEBI IDs" - tc_substrates_url
3) "All proteins in TCDB (FASTA format)" - tc_aa_url

In [2]:
def fetch_data(url):
    response = requests.get(url)
    response.raise_for_status()
    return response.text


def parse_data(uid_txt, substrates_txt, aa_txt):

    # TCID and Accesion ID (UID/RefSeq)
    uid_data = [line.split("\t") for line in uid_txt.strip().split("\n")]
    df_uid = pd.DataFrame(uid_data, columns=["UID", "TCID"])

    # Substrates (TCID, CHEBI ID and CHEBI Name)
    substrates_lines = substrates_txt.strip().split("\n")
    substrate_data = [[line.split("\t")[0], chebi.split(";")[0], chebi.split(";")[1]]
                        for line in substrates_lines
                        for chebi in line.split("\t")[1].split("|")]
    
    df_substrates = pd.DataFrame(substrate_data, columns=["TCID", "CHEBI ID", "CHEBI Name"])

    # AA sequence (TCID, UID and AA)
    fasta_io = StringIO(aa_txt)
    tc_data = [[record.description.split("|")[3].split()[0],  # TCID
                record.description.split("|")[2],  # UID
                str(record.seq)]  # AA
                for record in SeqIO.parse(fasta_io, "fasta")]
    
    df_aa = pd.DataFrame(tc_data, columns=["TCID", "UID", "AA"])

    return df_uid, df_substrates, df_aa

In [3]:
tc_uid_url = "https://www.tcdb.org/cgi-bin/projectv/public/acc2tcid.py"
tc_substrates_url = "https://www.tcdb.org/cgi-bin/substrates/getSubstrates.py"
tc_aa_url = "https://www.tcdb.org/public/tcdb"

tc_uid_txt = fetch_data(tc_uid_url)
tc_substrates_txt = fetch_data(tc_substrates_url)
tc_aa_txt = fetch_data(tc_aa_url)

df_uid, df_substrates, df_aa = parse_data(tc_uid_txt, tc_substrates_txt, tc_aa_txt)

First I'll merge the DFs.\
The relevant subclasses were retrieved in Misc/TCDB_composition.ipynb, and are: [1.A, 1.B, 1.C, 2.A, 3.A]\
From these subclasses, the ten most populated familes were obtained for further analysis. These, alongside their general mechanism and acting entity can be obtained from Misc/All_comp/family_mechanisms_entity_all.tsv.\
Only the top ten families in each of the subclasses above are of interest, but the rest will not be removed before the end.

In [4]:
df = pd.merge(df_uid, df_substrates, on="TCID", how="left")
df_aa = df_aa.drop("TCID", axis=1)
df = pd.merge(df, df_aa, left_on="UID", right_on="UID", how="left")


# Importing the families and related mechanisms
df_family_mechanisms = pd.read_csv("../Misc/All_comp/families_mechanisms_entity_final.tsv", sep="\t")
df_family_mechanisms["Mechanism"] = df_family_mechanisms["Mechanism"].replace({"â‡Œ": "=", "â†’": "="}, regex=True)

# Filter out families not in top 10 of each of the selected subclasses
df["Family"] = df["TCID"].apply(lambda x: ".".join(x.split(".")[:3]))
df = df.merge(df_family_mechanisms[["Family", "Mechanism", "Acting Entity"]], on="Family", how="left")
# Keeping the "Family"-column if only top ten families of each chosen subclass is to be kept in the end

Now, many of the CHEBI IDs are secondary IDs, and needs to be converted in order to map to Rhea for cross-checking.
Another hopeless finding, is the lack of correctness between ChEBI ID and ChEBI Name listed by TCDB. Therefore, this correction follows after s2p, and is denoted p2n (primary ID to name).

In [5]:
df_s2p = pd.read_csv("../ChEBI/s2p.tsv", sep="\t")
df_p2n = pd.read_csv("../ChEBI/p2n.tsv", sep="\t")

secondary_to_primary = dict(zip(df_s2p["Secondary_ID"], df_s2p["Primary_ID"]))
primary_to_name = dict(zip(df_p2n["Primary_ID"], df_p2n["ChEBI Name"]))

df["CHEBI ID"] = df["CHEBI ID"].apply(lambda x: secondary_to_primary.get(x, x))
df["CHEBI Name"] = df.apply(lambda row: primary_to_name.get(row["CHEBI ID"], row["CHEBI Name"]), axis=1)

In [ ]:
df_p2n = pd.read_csv("../ChEBI/p2n.tsv", sep="\t")
primary_to_name = dict(zip(df_p2n["Primary_ID"], df_p2n["ChEBI Name"]))

df["CHEBI Name"] = df.apply(lambda row: primary_to_name.get(row["CHEBI ID"], row["CHEBI Name"]), axis=1)

Future plan: Go through families_mechanisms_all.tsv and create another column for the acting entity in the reaction. The acting entity x will be marked as such: {x}\
This was first conudcted through AI, as this is a tedious manual task, before it was verified and edited by hand.

In [7]:
def create_reaction_row(row):

    if pd.isna(row["Acting Entity"]) or pd.isna(row["CHEBI Name"]):
        return row["Mechanism"]
    

    mechanisms = row["Mechanism"].split(", ")
    acting_entities = str(row["Acting Entity"]).split(", ")
    chebi_name = str(row["CHEBI Name"])

    reactions = []
    for mechanism, entity in zip(mechanisms, acting_entities):
        reaction = mechanism.replace(entity, chebi_name)
        reactions.append(reaction)
    
    return ", ".join(reactions)

df["Reaction"] = df.apply(create_reaction_row, axis=1)

Now the next focus will be to obtain the Rhea reaction IDs, including both the names and the ChEBI IDs. This will go through UniProt.\
Starting off with mapping the correct Rhea IDs (RID) through a UniProt SPARQLE-query, saved as UniProt/Modified_queries/UID_RID.tsv.\
The complete data to the RID will be filled in at the end.

In [8]:
uid_rhea_map = pd.read_csv("../UniProt/Modified_queries/RID_UID.tsv", sep="\t")

df = df.merge(uid_rhea_map, left_on="UID", right_on="UID", how="left")
df["RID"] = df["RID"].apply(lambda x: f"RHEA:{int(x)}" if pd.notnull(x) else x)

Many of the UIDs are refseq IDs (RSIDs), and hence unattainable through UniProt-Rhea mapping. Therefore, a query was written, and can be found in UniProt/UniProt_Rhea.ipynb as query2_new. This obtains the UID and RSID whenever attainable. The query was run online on https://sparql.uniprot.org/, and the result is stored in UniProt/Modified_queries/query2_new.csv. After a second of thought, I realize that this file is too large to push to Git, so this must be created manually. The notebook modifies the csv as wanted, easing the use here. The same goes for the modified file, UID_RSID.tsv is close to the max push size, hence not included in the repo.

Now, a new column is created AID (Accession ID), in order to retain the elements of the UID-column that does not have an RID. Following, these will be attempted converted into UIDs, assuming many of them are RSIDs. Then an RID will be attributed.

In [9]:
refseq = pd.read_csv("../UniProt/Modified_queries/UID_RSID.tsv", sep="\t")

df["AID"] = df["UID"].where(df["RID"].isna())
df = df.merge(refseq, left_on="AID", right_on="RSID", how="left")
df = df.rename(columns={"UID_x": "UID", "UID_y": "UID2"})
df = df.drop(columns=["RSID"])

Now mapping UID2-values to RIDs. Then the two RID-columns are merged into one.

In [10]:
df = df.merge(uid_rhea_map, left_on="UID2", right_on="UID", how="left")
df["RID_y"] = df["RID_y"].apply(lambda x: f"RHEA:{int(x)}" if pd.notnull(x) else x)
df["RID"] = df["RID_x"].combine_first(df["RID_y"])
df = df.drop(columns=["UID_y", "RID_x", "RID_y"])
df.rename(columns={"UID_x": "UID"}, inplace=True)

Now, it is time to fill up the new RIDs with their data!\
The following step is to include the mapped reaction data for each RID. Both the equation, the ChEBI IDs, the ChEBI Name and EC number. This is stored in the columns titled as such: R:{col_name}\
The Rhea data is obtained from rhea-db.org 27.01.25, and can be found in the Rhea-folder.

In [11]:
rhea = pd.read_csv("../Rhea/Rhea.tsv", sep="\t")

rhea["Reaction identifier"] = rhea["Reaction identifier"].astype(str)
df["RID"] = df["RID"].astype(str)
df = df.merge(rhea, left_on="RID", right_on="Reaction identifier", how="left")

df = df.drop(columns=["Reaction identifier"])
df = df.rename(columns=lambda x: f"R:{x}" if x not in
               ["TCID", "UID", "AA", "CHEBI ID", "CHEBI Name", "Mechanism", 
                "Acting Entity", "Reaction", "RID", "Family", "AID", "UID2"]
                else x)
df["RID"] = df["RID"].replace("nan", np.nan)

Now, it looks as desired!\
Down below is only some minor fashionable adjustments, and laying UID2 over UID to only have one UID column. This now contains all the UIDs of what was previously RSIDs, plus the RSIDs that could not be converted. Also the few instances (<100) that got no AA attributed, is removed, as this is useless for the further applications.

In [ ]:
cols = df.columns.tolist()
cols.insert(1, cols.pop(cols.index("UID2")))
df = df[cols]
df["UID"] = df["UID2"].combine_first(df["UID"])
df = df.drop(columns=["AID", "UID2"])
df.dropna(subset=["AA"], inplace=True)

In the end, a fasta-file is created, as this can be used for the BLAST. Similiarily, a .tsv is created, as this enables the merging of the final results from the BLAST.

In [13]:
df_fasta = df.drop_duplicates(subset=["UID", "TCID", "AA"])
fasta_file = "transporters1.fasta"

with open(fasta_file, "w") as f:
    for index, row in df_fasta.iterrows():
        uid = row["UID"]
        tcid = row["TCID"]
        sequence = row["AA"]
        f.write(f">{uid}|{tcid}\n{sequence}\n")

df.to_csv("transporters1_df.tsv", sep="\t", index=False)